# Pitch Arsenal Analysis

A deep dive into how each pitcher's arsenal compares:
- Velocity, movement, spin rate by pitch type
- Release point and extension
- Effective velocity differentials

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = 'data'
PITCHERS = {
    'Ohtani': 'shohei_ohtani.csv',
    'Sanchez': 'cristopher_sanchez.csv',
    'Misiorowski': 'jacob_misiorowski.csv'
}

dfs = {}
for name, fname in PITCHERS.items():
    df = pd.read_csv(f'{DATA_DIR}/{fname}')
    df['player_name_clean'] = df['player_name'].str.replace('\n', ', ', regex=False)
    dfs[name] = df

In [ ]:
# Detailed pitch-type comparison
summary_rows = []
for name, df in dfs.items():
    for pt in sorted(df['pitch_type'].dropna().unique()):
        sub = df[df['pitch_type'] == pt]
        n = len(sub)
        if n < 10:
            continue
        row = {
            'Pitcher': name, 'Pitch': pt, 'Count': n,
            'Usage%': round(n / len(df) * 100, 1),
            'Velo': round(sub['release_speed'].mean(), 1),
            'VeloSD': round(sub['release_speed'].std(), 1),
            'Spin': round(sub['release_spin_rate'].mean(), 0),
            'HMov': round(sub['pfx_x'].mean(), 1),
            'VMov': round(sub['pfx_z'].mean(), 1),
            'Extension': round(sub['release_extension'].mean(), 2),
        }
        summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
print('PITCH ARSENAL COMPARISON')
print('=' * 120)
print(summary_df.to_string(index=False))
summary_df.to_csv('figures/pitch_arsenal_comparison.csv', index=False)
print('\nSaved to figures/pitch_arsenal_comparison.csv')

In [ ]:
# Velocity by pitch type
fig, ax = plt.subplots(figsize=(12, 6))
colors = {'Ohtani': '#1f77b4', 'Sanchez': '#ff7f0e', 'Misiorowski': '#2ca02c'}
markers = {'Ohtani': 'o', 'Sanchez': 's', 'Misiorowski': 'D'}

pitch_order = ['FF', 'SI', 'FC', 'SL', 'ST', 'CU', 'CH', 'FS']
visible_pitches = [pt for pt in pitch_order
                    if any(pt in dfs[n]['pitch_type'].dropna().unique() for n in dfs)]

for idx, pt in enumerate(visible_pitches):
    for name in dfs:
        sub = dfs[name][dfs[name]['pitch_type'] == pt]
        vel = sub['release_speed'].dropna()
        if len(vel) < 5:
            continue
        offset = {'Ohtani': -0.2, 'Sanchez': 0, 'Misiorowski': 0.2}[name]
        x_pos = idx + offset
        ax.errorbar(x_pos, vel.mean(), yerr=vel.std(), fmt=markers[name],
                   color=colors[name], capsize=4, capthick=2, markersize=10,
                   label=name if idx == 0 else '')

ax.set_xticks(range(len(visible_pitches)))
ax.set_xticklabels(visible_pitches)
ax.set_ylabel('Release Speed (mph)')
ax.set_title('Velocity by Pitch Type (Mean +/- SD)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('figures/velocity_by_pitch.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Spin rate comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, df) in zip(axes, dfs.items()):
    pitch_types = sorted(df['pitch_type'].dropna().unique())
    data = [df[df['pitch_type'] == pt]['release_spin_rate'].dropna().values for pt in pitch_types]
    bp = ax.boxplot(data, label=pitch_types, patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Set2(np.linspace(0, 1, len(pitch_types)))):
        patch.set_facecolor(color)
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_ylabel('Spin Rate (rpm)')
plt.suptitle('Spin Rate Distribution by Pitch Type', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/spin_rate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Whiff rate by pitch type
print('WHIFF RATE BY PITCH TYPE')
print('=' * 80)
whiff_rows = []
for name, df in dfs.items():
    for pt in sorted(df['pitch_type'].dropna().unique()):
        sub = df[df['pitch_type'] == pt]
        swings = sub[sub['description'].str.contains('swinging_strike|hit_into_play|foul', na=False, regex=True)]
        whiffs = sub[sub['description'].str.contains('swinging_strike', na=False)]
        n_swings = len(swings)
        n_whiffs = len(whiffs)
        whiff_pct = n_whiffs / n_swings * 100 if n_swings > 0 else 0
        whiff_rows.append({'Pitcher': name, 'Pitch': pt, 'Swings': n_swings,
                          'Whiffs': n_whiffs, 'Whiff%': f'{whiff_pct:.1f}%'})
whiff_df = pd.DataFrame(whiff_rows)
print(whiff_df.to_string(index=False))
print('\n\nOVERALL WHIFF RATE')
for name, df in dfs.items():
    swings = df[df['description'].str.contains('swinging_strike|hit_into_play|foul', na=False, regex=True)]
    whiffs = df[df['description'].str.contains('swinging_strike', na=False)]
    print(f'{name}: {len(whiffs)}/{len(swings)} = {len(whiffs)/len(swings)*100:.1f}%')